<a href="https://colab.research.google.com/github/Nandini-9908/ml_project/blob/main/ECG_Arrhythmia_Detection_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ECG Arrhythmia Detection — MIT-BIH (v2: Practical, Deployable Version)

**Goal:** given *any* ECG record, automatically detect every heartbeat and classify it as **Normal** or one of the **Abnormal** types (S / V / F), so it could realistically sit behind a Holter monitor or wearable.

**What's new vs. v1** (fixes the near-zero S/F recall from the first attempt):
1. **RR-interval features** — the model now sees beat *timing* (previous RR, next RR, local RR ratio), not just beat *shape*. This is what actually separates S-class beats from Normal.
2. **Oversampling of minority classes** — S/F/Q beats are duplicated with small jitter/noise so the model sees them as often as N during training, instead of relying on loss-weighting alone.
3. **Focal loss** — focuses training on hard/rare examples instead of the easy majority class.
4. **A single ready-to-use function**: `predict_record(record_name)` — point it at any MIT-BIH record (or your own similarly-formatted signal) and get back detected beats, per-beat Normal/Abnormal classification, a plot, and summary stats.

In [2]:
# 1. Install dependencies (Colab)
!pip install -q wfdb tensorflow scikit-learn matplotlib seaborn

In [ ]:
import wfdb
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import butter, filtfilt
from collections import Counter

DATA_DIR = "mitdb"
os.makedirs(DATA_DIR, exist_ok=True)

RECORD_NAMES = [
    '100','101','102','103','104','105','106','107','108','109',
    '111','112','113','114','115','116','117','118','119','121',
    '122','123','124','200','201','202','203','205','207','208',
    '209','210','212','213','214','215','217','219','220','221',
    '222','223','228','230','231','232','233','234'
]

def download_records(record_names, data_dir=DATA_DIR):
    for rec in record_names:
        local_path = os.path.join(data_dir, rec + ".dat")
        if not os.path.exists(local_path):
            wfdb.dl_database('mitdb', dl_dir=data_dir, records=[rec])
    print(f"Downloaded/verified {len(record_names)} records.")

RECORDS_TO_USE = RECORD_NAMES  # use RECORD_NAMES[:10] for a quick test run first
download_records(RECORDS_TO_USE)

Generating record list for: 100
Generating list of all files for: 100
Finished downloading files
Generating record list for: 101
Generating list of all files for: 101
Finished downloading files
Generating record list for: 102
Generating list of all files for: 102
Finished downloading files
Generating record list for: 103
Generating list of all files for: 103
Finished downloading files
Generating record list for: 104
Generating list of all files for: 104
Finished downloading files
Generating record list for: 105
Generating list of all files for: 105
Finished downloading files
Generating record list for: 106
Generating list of all files for: 106
Finished downloading files
Generating record list for: 107
Generating list of all files for: 107
Finished downloading files
Generating record list for: 108
Generating list of all files for: 108
Finished downloading files
Generating record list for: 109
Generating list of all files for: 109
Finished downloading files
Generating record list for: 11

KeyboardInterrupt: 

## 2. Preprocessing

In [ ]:
FS = 360  # MIT-BIH sampling rate (Hz)

def bandpass_filter(signal, lowcut=0.5, highcut=40.0, fs=FS, order=2):
    nyq = 0.5 * fs
    low, high = lowcut / nyq, highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)

def normalize(signal):
    return (signal - np.mean(signal)) / (np.std(signal) + 1e-8)

def load_record(rec_name, data_dir=DATA_DIR):
    record = wfdb.rdrecord(os.path.join(data_dir, rec_name))
    annotation = wfdb.rdann(os.path.join(data_dir, rec_name), 'atr')
    signal = record.p_signal[:, 0]  # MLII lead
    filtered = normalize(bandpass_filter(signal))
    return filtered, annotation

## 2b. Preprocessing Applied to Every Record (Verification)
`load_record()` already applies bandpass filtering + normalization to **every** record when `build_dataset()` loops over them — this section just makes that visible and lets you sanity-check it record by record before trusting the downstream pipeline.

For each of the 48 records we report: raw signal duration, mean/std before and after preprocessing, and the number of expert-annotated beats. Then we plot a short raw-vs-filtered comparison for a couple of sample records.

In [ ]:
import pandas as pd

def preprocess_summary(record_names, data_dir=DATA_DIR):
    rows = []
    for rec in record_names:
        record = wfdb.rdrecord(os.path.join(data_dir, rec))
        annotation = wfdb.rdann(os.path.join(data_dir, rec), 'atr')
        raw = record.p_signal[:, 0]

        filtered = bandpass_filter(raw)
        processed = normalize(filtered)

        rows.append({
            'record': rec,
            'duration_min': round(len(raw) / FS / 60, 2),
            'raw_mean': round(np.mean(raw), 4),
            'raw_std': round(np.std(raw), 4),
            'processed_mean': round(np.mean(processed), 4),
            'processed_std': round(np.std(processed), 4),
            'n_annotated_beats': len(annotation.sample),
        })
    return pd.DataFrame(rows)

preprocess_df = preprocess_summary(RECORDS_TO_USE)
print(f"Preprocessed {len(preprocess_df)} records.")
print(f"Total annotated beats across dataset: {preprocess_df['n_annotated_beats'].sum()}")
preprocess_df

In [ ]:
# Visual sanity check: raw vs. filtered+normalized signal, first 5 seconds, for a couple of records
sample_records_to_plot = RECORDS_TO_USE[:2]

fig, axes = plt.subplots(len(sample_records_to_plot), 2, figsize=(14, 3 * len(sample_records_to_plot)))
if len(sample_records_to_plot) == 1:
    axes = axes.reshape(1, -1)

for row, rec in enumerate(sample_records_to_plot):
    record = wfdb.rdrecord(os.path.join(DATA_DIR, rec))
    raw = record.p_signal[:, 0]
    processed = normalize(bandpass_filter(raw))

    n_samples = 5 * FS
    t = np.arange(n_samples) / FS

    axes[row, 0].plot(t, raw[:n_samples], color='gray', linewidth=0.8)
    axes[row, 0].set_title(f'Record {rec} — Raw')
    axes[row, 0].set_xlabel('Time (s)')

    axes[row, 1].plot(t, processed[:n_samples], color='steelblue', linewidth=0.8)
    axes[row, 1].set_title(f'Record {rec} — Filtered + Normalized')
    axes[row, 1].set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

## 3. Beat Detection — Pan-Tompkins

In [ ]:
def pan_tompkins_detector(signal, fs=FS):
    diff = np.diff(signal, prepend=signal[0])
    squared = diff ** 2
    window_size = int(0.15 * fs)
    integrated = np.convolve(squared, np.ones(window_size) / window_size, mode='same')
    threshold = np.mean(integrated) + 0.5 * np.std(integrated)
    min_distance = int(0.2 * fs)

    peaks = []
    i = 1
    while i < len(integrated) - 1:
        if integrated[i] > threshold and integrated[i] > integrated[i-1] and integrated[i] >= integrated[i+1]:
            peaks.append(i)
            i += min_distance
        else:
            i += 1
    return np.array(peaks)

## 4. Beat Segmentation + RR-Interval Features + AAMI Labels

Each beat now becomes **two things fed to the model**:
- A **200-sample waveform window** around the R-peak (shape)
- **3 RR-interval features**: previous RR interval, next RR interval, and the ratio between them (timing/rhythm context)

This timing context is exactly what a simple shape-only CNN was missing for S-class beats.

In [ ]:
AAMI_MAP = {
    'N': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N',
    'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S',
    'V': 'V', 'E': 'V',
    'F': 'F',
    'P': 'Q', '/': 'Q', 'f': 'Q', 'Q': 'Q', 'U': 'Q'
}
CLASSES = ['N', 'S', 'V', 'F', 'Q']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
WINDOW_BEFORE, WINDOW_AFTER = 90, 110

def extract_beats_with_rr(signal, annotation, fs=FS):
    samples = annotation.sample
    symbols = annotation.symbol

    beats, rr_features, labels = [], [], []
    for idx in range(1, len(samples) - 1):  # need a beat before and after for RR features
        symbol = symbols[idx]
        if symbol not in AAMI_MAP:
            continue
        sample = samples[idx]
        start, end = sample - WINDOW_BEFORE, sample + WINDOW_AFTER
        if start < 0 or end > len(signal):
            continue

        prev_rr = (samples[idx] - samples[idx - 1]) / fs
        next_rr = (samples[idx + 1] - samples[idx]) / fs
        rr_ratio = prev_rr / next_rr if next_rr > 0 else 1.0

        beats.append(signal[start:end])
        rr_features.append([prev_rr, next_rr, rr_ratio])
        labels.append(CLASS_TO_IDX[AAMI_MAP[symbol]])

    return np.array(beats), np.array(rr_features), np.array(labels)

def build_dataset(record_names, data_dir=DATA_DIR):
    all_beats, all_rr, all_labels = [], [], []
    for rec in record_names:
        sig, ann = load_record(rec, data_dir)
        beats, rr, labels = extract_beats_with_rr(sig, ann)
        all_beats.append(beats)
        all_rr.append(rr)
        all_labels.append(labels)
    X = np.concatenate(all_beats, axis=0)
    RR = np.concatenate(all_rr, axis=0)
    y = np.concatenate(all_labels, axis=0)
    return X, RR, y

X, RR, y = build_dataset(RECORDS_TO_USE)
X = X[..., np.newaxis]
print("Waveform shape:", X.shape, "| RR features shape:", RR.shape)
print("Class distribution:", {CLASSES[k]: v for k, v in Counter(y).items()})

## 5. Inter-Patient Train/Test Split (DS1/DS2)
Same rigorous split as before — certain patients go entirely to train, others entirely to test, so the model can't just memorize a patient's beat shape.

In [ ]:
DS1 = ['101','106','108','109','112','114','115','116','118','119',
       '122','124','201','203','205','207','208','209','215','220',
       '223','230']
DS2 = ['100','103','105','111','113','117','121','123','200','202',
       '210','212','213','214','219','221','222','228','231','232',
       '233','234']

train_records = [r for r in RECORDS_TO_USE if r in DS1]
test_records = [r for r in RECORDS_TO_USE if r in DS2]

if len(train_records) == 0 or len(test_records) == 0:
    from sklearn.model_selection import train_test_split
    X_train, X_test, RR_train, RR_test, y_train, y_test = train_test_split(
        X, RR, y, test_size=0.2, stratify=y, random_state=42)
else:
    X_train, RR_train, y_train = build_dataset(train_records)
    X_test, RR_test, y_test = build_dataset(test_records)
    X_train, X_test = X_train[..., np.newaxis], X_test[..., np.newaxis]

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train class distribution:", {CLASSES[k]: v for k, v in Counter(y_train).items()})

## 6. Oversample Minority Classes (Train Set Only)
We duplicate S/V/F/Q beats — with small random jitter/noise added to the waveform so duplicates aren't exact copies — until every class has a reasonable share of the training set. **Never oversample the test set** — that would fake your evaluation.

In [ ]:
def oversample(X, RR, y, target_ratio=0.5, seed=42):
    """Oversample each minority class up to target_ratio * majority_class_count."""
    rng = np.random.default_rng(seed)
    counts = Counter(y)
    majority_count = max(counts.values())
    target_count = int(majority_count * target_ratio)

    X_out, RR_out, y_out = [X], [RR], [y]
    for cls, count in counts.items():
        if count >= target_count:
            continue
        n_needed = target_count - count
        cls_idx = np.where(y == cls)[0]
        chosen = rng.choice(cls_idx, size=n_needed, replace=True)

        waveform_noise = rng.normal(0, 0.02, size=(n_needed,) + X.shape[1:])
        rr_noise = rng.normal(0, 0.01, size=(n_needed,) + RR.shape[1:])

        X_out.append(X[chosen] + waveform_noise)
        RR_out.append(RR[chosen] + rr_noise)
        y_out.append(y[chosen])

    X_final = np.concatenate(X_out, axis=0)
    RR_final = np.concatenate(RR_out, axis=0)
    y_final = np.concatenate(y_out, axis=0)

    shuffle_idx = rng.permutation(len(y_final))
    return X_final[shuffle_idx], RR_final[shuffle_idx], y_final[shuffle_idx]

X_train_os, RR_train_os, y_train_os = oversample(X_train, RR_train, y_train, target_ratio=0.5)
print("After oversampling:", {CLASSES[k]: v for k, v in Counter(y_train_os).items()})

## 7. Model — Two-Branch CNN (Waveform + RR Features)
Branch A processes the beat shape (CNN). Branch B processes the 3 RR-interval numbers (dense). They're concatenated before the final classification layer — so the model can use both **what the beat looks like** and **how the heart's rhythm is behaving around it**.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model

def build_model(waveform_shape=(200, 1), rr_shape=(3,), n_classes=5):
    # Branch A: waveform (CNN)
    wave_input = layers.Input(shape=waveform_shape, name='waveform')
    x = layers.Conv1D(16, 7, activation='relu', padding='same')(wave_input)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(32, 5, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)

    # Branch B: RR-interval features (dense)
    rr_input = layers.Input(shape=rr_shape, name='rr_features')
    r = layers.Dense(16, activation='relu')(rr_input)
    r = layers.Dense(16, activation='relu')(r)

    combined = layers.Concatenate()([x, r])
    d = layers.Dense(32, activation='relu')(combined)
    d = layers.Dropout(0.3)(d)
    output = layers.Dense(n_classes, activation='softmax')(d)

    return Model(inputs=[wave_input, rr_input], outputs=output)

model = build_model()
model.summary()

## 8. Focal Loss
Standard cross-entropy treats every mistake equally. Focal loss down-weights easy/majority examples (correctly-classified N beats) and focuses gradient on hard/rare examples (S, F) — this is the standard fix for severe class imbalance like MIT-BIH's.

In [ ]:
def sparse_categorical_focal_loss(gamma=2.0, alpha=None):
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        y_true_one_hot = tf.one_hot(y_true, depth=tf.shape(y_pred)[-1])

        ce = -y_true_one_hot * tf.math.log(y_pred)
        weight = tf.pow(1 - y_pred, gamma)
        fl = weight * ce

        if alpha is not None:
            alpha_tensor = tf.constant(alpha, dtype=tf.float32)
            fl = fl * alpha_tensor

        return tf.reduce_sum(fl, axis=-1)
    return loss_fn

# Give extra weight to rare classes (S, F, Q) inside the focal loss itself
class_counts = Counter(y_train_os)
total = sum(class_counts.values())
alpha = [total / (len(CLASSES) * class_counts.get(i, 1)) for i in range(len(CLASSES))]
alpha = [a / max(alpha) for a in alpha]  # normalize to max 1.0
print("Focal loss alpha per class:", dict(zip(CLASSES, [round(a, 3) for a in alpha])))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=sparse_categorical_focal_loss(gamma=2.0, alpha=alpha),
    metrics=['accuracy']
)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True, monitor='val_loss'),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
]

history = model.fit(
    {'waveform': X_train_os, 'rr_features': RR_train_os}, y_train_os,
    validation_split=0.1,
    epochs=40,
    batch_size=128,
    callbacks=callbacks
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_title('Accuracy'); axes[1].legend()
plt.tight_layout()
plt.show()

## 9. Evaluation — on the untouched (non-oversampled) test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_probs = model.predict({'waveform': X_test, 'rr_features': RR_test})
y_pred = np.argmax(y_pred_probs, axis=1)

print(classification_report(y_test, y_pred, target_names=CLASSES, digits=4))

cm = confusion_matrix(y_test, y_pred)

def per_class_sensitivity_specificity(cm):
    n_classes = cm.shape[0]
    results = {}
    total = cm.sum()
    for i in range(n_classes):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = total - TP - FN - FP
        sens = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        spec = TN / (TN + FP) if (TN + FP) > 0 else 0.0
        results[CLASSES[i]] = {'sensitivity': sens, 'specificity': spec}
    return results

metrics = per_class_sensitivity_specificity(cm)
for cls, vals in metrics.items():
    print(f"{cls}: sensitivity={vals['sensitivity']:.4f}  specificity={vals['specificity']:.4f}")

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES, cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix (v2)')
plt.show()

normal_idx = CLASS_TO_IDX['N']
false_alarms = np.sum((y_pred != normal_idx) & (y_test == normal_idx))
total_minutes = len(test_records) * 30 if 'test_records' in dir() and test_records else (len(y_test) / FS) / 60
print(f"\nFalse alarms per minute: {false_alarms / total_minutes:.3f}")

macro_f1 = classification_report(y_test, y_pred, target_names=CLASSES, output_dict=True)['macro avg']['f1-score']
print(f"Macro-F1: {macro_f1:.4f}")

## 9b. Comparing Multiple Models
The two-branch CNN above is one option — here we train **three more architectures** on the exact same data (oversampled train set, untouched test set) so you can compare which works best for this problem, rather than relying on a single model.

1. **Waveform-only CNN** — shape only, no RR features (this reproduces the v1-style baseline for direct comparison)
2. **Two-branch CNN + RR** — the model already trained above (Section 7-8)
3. **BiLSTM + RR** — treats the beat waveform as a sequence rather than a fixed shape, sometimes captures morphology differently than a CNN
4. **Random Forest on hand-crafted features** — a classical ML baseline using statistical features (mean, std, skew, kurtosis, min, max) + RR features, no deep learning at all

All four are evaluated with the same metrics so you can pick the best trade-off between accuracy and deployability (the Random Forest doesn't convert to TFLite the same way, so keep that in mind if edge deployment is the priority).

In [ ]:
# --- Model A: Waveform-only CNN (no RR features) ---
def build_waveform_only_model(waveform_shape=(200, 1), n_classes=5):
    inp = layers.Input(shape=waveform_shape)
    x = layers.Conv1D(16, 7, activation='relu', padding='same')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(32, 5, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)
    return Model(inputs=inp, outputs=out)

model_waveform_only = build_waveform_only_model()
model_waveform_only.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=sparse_categorical_focal_loss(gamma=2.0, alpha=alpha),
    metrics=['accuracy']
)
history_a = model_waveform_only.fit(
    X_train_os, y_train_os,
    validation_split=0.1, epochs=40, batch_size=128,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True, monitor='val_loss'),
               tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)],
    verbose=0
)
print("Model A (waveform-only CNN) training complete.")

In [ ]:
# --- Model C: BiLSTM + RR features ---
def build_bilstm_model(waveform_shape=(200, 1), rr_shape=(3,), n_classes=5):
    wave_input = layers.Input(shape=waveform_shape, name='waveform')
    x = layers.Bidirectional(layers.LSTM(32, return_sequences=True))(wave_input)
    x = layers.Bidirectional(layers.LSTM(16))(x)

    rr_input = layers.Input(shape=rr_shape, name='rr_features')
    r = layers.Dense(16, activation='relu')(rr_input)

    combined = layers.Concatenate()([x, r])
    d = layers.Dense(32, activation='relu')(combined)
    d = layers.Dropout(0.3)(d)
    out = layers.Dense(n_classes, activation='softmax')(d)
    return Model(inputs=[wave_input, rr_input], outputs=out)

model_bilstm = build_bilstm_model()
model_bilstm.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=sparse_categorical_focal_loss(gamma=2.0, alpha=alpha),
    metrics=['accuracy']
)
history_c = model_bilstm.fit(
    {'waveform': X_train_os, 'rr_features': RR_train_os}, y_train_os,
    validation_split=0.1, epochs=40, batch_size=128,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True, monitor='val_loss'),
               tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)],
    verbose=0
)
print("Model C (BiLSTM + RR) training complete. Note: LSTMs are slower to train than CNNs — this cell may take a while.")

In [ ]:
# --- Model D: Random Forest on hand-crafted features (classical ML baseline) ---
from scipy.stats import skew, kurtosis
from sklearn.ensemble import RandomForestClassifier

def extract_handcrafted_features(waveforms, rr_feats):
    feats = []
    for wave in waveforms:
        wave = wave.flatten()
        feats.append([
            np.mean(wave), np.std(wave), np.min(wave), np.max(wave),
            skew(wave), kurtosis(wave), np.ptp(wave)
        ])
    feats = np.array(feats)
    return np.concatenate([feats, rr_feats], axis=1)

X_train_rf = extract_handcrafted_features(X_train_os, RR_train_os)
X_test_rf = extract_handcrafted_features(X_test, RR_test)

rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=15, class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf_model.fit(X_train_rf, y_train_os)
print("Model D (Random Forest) training complete.")

In [ ]:
# --- Compare all four models on the SAME untouched test set ---
def evaluate_model(name, y_true, y_pred):
    report = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True, zero_division=0)
    cm_local = confusion_matrix(y_true, y_pred)
    sens_spec = per_class_sensitivity_specificity(cm_local)

    normal_idx_local = CLASS_TO_IDX['N']
    fa = np.sum((y_pred != normal_idx_local) & (y_true == normal_idx_local))
    fa_per_min_local = fa / total_minutes

    row = {'model': name, 'macro_f1': report['macro avg']['f1-score'],
           'false_alarms_per_min': fa_per_min_local}
    for cls in CLASSES:
        row[f'{cls}_sensitivity'] = sens_spec[cls]['sensitivity']
    return row

# Predictions from each model
pred_a = np.argmax(model_waveform_only.predict(X_test, verbose=0), axis=1)
pred_b = y_pred  # already computed in Section 9 for the two-branch CNN
pred_c = np.argmax(model_bilstm.predict({'waveform': X_test, 'rr_features': RR_test}, verbose=0), axis=1)
pred_d = rf_model.predict(X_test_rf)

comparison_rows = [
    evaluate_model('A: Waveform-only CNN', y_test, pred_a),
    evaluate_model('B: Two-branch CNN + RR', y_test, pred_b),
    evaluate_model('C: BiLSTM + RR', y_test, pred_c),
    evaluate_model('D: Random Forest (handcrafted)', y_test, pred_d),
]

comparison_df = pd.DataFrame(comparison_rows).set_index('model')
comparison_df = comparison_df.round(4)
comparison_df

In [ ]:
# Visual comparison of macro-F1 across models
plt.figure(figsize=(8, 4))
plt.bar(comparison_df.index, comparison_df['macro_f1'], color=['#4C72B0', '#55A868', '#C44E52', '#8172B2'])
plt.ylabel('Macro-F1')
plt.title('Model Comparison — Macro-F1 on Test Set')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

best_model_name = comparison_df['macro_f1'].idxmax()
print(f"Best model by macro-F1: {best_model_name}")

**Note:** `predict_record()` in Section 11 currently uses `model` (the two-branch CNN + RR, Model B). If the comparison above shows a different model performing better, swap the `model.predict(...)` call inside `predict_record()` for the winning model's `.predict(...)` call — for the BiLSTM that's a drop-in swap since it takes the same two inputs; for the Random Forest you'd need to also call `extract_handcrafted_features()` on the detected beats before predicting.

## 10. Binary View — Normal vs Abnormal
Since the practical ask is *"predict whether a heartbeat is normal or abnormal"*, here's that simplified, clinically-relevant view collapsed from the 5-class output.

In [ ]:
y_test_binary = (y_test != CLASS_TO_IDX['N']).astype(int)     # 0=Normal, 1=Abnormal
y_pred_binary = (y_pred != CLASS_TO_IDX['N']).astype(int)

print(classification_report(y_test_binary, y_pred_binary, target_names=['Normal', 'Abnormal'], digits=4))

cm_bin = confusion_matrix(y_test_binary, y_pred_binary)
plt.figure(figsize=(4, 4))
sns.heatmap(cm_bin, annot=True, fmt='d', xticklabels=['Normal','Abnormal'],
            yticklabels=['Normal','Abnormal'], cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Normal vs Abnormal')
plt.show()

## 11. Ready-to-Use Inference Function
This is the practical deliverable: **give it any record name, and it detects every beat, classifies each one, plots the result, and prints a percentage breakdown for every beat type (N, S, V, F, Q).** Works on any MIT-BIH record (or your own signal in the same format — see the notes below).

In [ ]:
def predict_record(record_name, data_dir=DATA_DIR, plot_seconds=10, plot_start_sec=0):
    """
    Full end-to-end pipeline for ONE record:
      1. Load + preprocess the signal
      2. Detect beats with Pan-Tompkins
      3. Compute RR-interval features per detected beat
      4. Classify each beat (5-class + binary Normal/Abnormal)
      5. Plot a segment of the ECG with predictions
      6. Return a summary

    If the record isn't downloaded yet, it will be fetched from PhysioNet automatically.
    """
    if not os.path.exists(os.path.join(data_dir, record_name + ".dat")):
        wfdb.dl_database('mitdb', dl_dir=data_dir, records=[record_name])

    signal, _ = load_record(record_name, data_dir)
    peaks = pan_tompkins_detector(signal)

    beats, rr_feats, valid_peaks = [], [], []
    for i in range(1, len(peaks) - 1):
        p = peaks[i]
        start, end = p - WINDOW_BEFORE, p + WINDOW_AFTER
        if start < 0 or end > len(signal):
            continue
        prev_rr = (peaks[i] - peaks[i - 1]) / FS
        next_rr = (peaks[i + 1] - peaks[i]) / FS
        rr_ratio = prev_rr / next_rr if next_rr > 0 else 1.0

        beats.append(signal[start:end])
        rr_feats.append([prev_rr, next_rr, rr_ratio])
        valid_peaks.append(p)

    if len(beats) == 0:
        print("No valid beats detected.")
        return None

    beats = np.array(beats)[..., np.newaxis]
    rr_feats = np.array(rr_feats)

    probs = model.predict({'waveform': beats, 'rr_features': rr_feats}, verbose=0)
    pred_classes = np.argmax(probs, axis=1)
    pred_labels = [CLASSES[c] for c in pred_classes]
    is_abnormal = pred_classes != CLASS_TO_IDX['N']

    # --- Summary ---
    total_beats = len(valid_peaks)
    class_counts = Counter(pred_labels)
    # Percentage for every beat type, including classes with zero detections
    class_percentages = {
        cls: round(100 * class_counts.get(cls, 0) / total_beats, 2)
        for cls in CLASSES
    }
    summary = {
        'record': record_name,
        'total_beats_detected': total_beats,
        'class_counts': {cls: class_counts.get(cls, 0) for cls in CLASSES},
        'class_percentages': class_percentages,
        'abnormal_beat_count': int(is_abnormal.sum()),
        'abnormal_beat_pct': round(100 * is_abnormal.mean(), 2),
        'peak_samples': valid_peaks,
        'predicted_labels': pred_labels,
    }

    print(f"Record: {record_name}")
    print(f"Total beats detected: {summary['total_beats_detected']}\n")
    print(f"{'Beat type':<12}{'Count':<10}{'Percentage':<12}")
    print("-" * 34)
    for cls in CLASSES:
        count = summary['class_counts'][cls]
        pct = summary['class_percentages'][cls]
        print(f"{cls:<12}{count:<10}{pct}%")
    print("-" * 34)
    print(f"{'Abnormal':<12}{summary['abnormal_beat_count']:<10}{summary['abnormal_beat_pct']}%")

    # --- Plot a segment ---
    start_sample = int(plot_start_sec * FS)
    end_sample = int((plot_start_sec + plot_seconds) * FS)
    segment = signal[start_sample:end_sample]
    time_axis = np.arange(len(segment)) / FS

    plt.figure(figsize=(15, 4))
    plt.plot(time_axis, segment, color='steelblue', linewidth=0.8)
    for p, lbl in zip(valid_peaks, pred_labels):
        if start_sample <= p < end_sample:
            rel_t = (p - start_sample) / FS
            color = 'green' if lbl == 'N' else 'red'
            plt.axvline(rel_t, color=color, alpha=0.3, linewidth=2)
            plt.text(rel_t, max(segment) * 1.05, lbl, ha='center', color=color,
                      fontsize=8, fontweight='bold')
    plt.xlabel('Time (s)'); plt.ylabel('Normalized amplitude')
    plt.title(f'Record {record_name} — green=Normal, red=Abnormal')
    plt.tight_layout()
    plt.show()

    return summary

# --- Example usage: run on any record ---
summary = predict_record('232', plot_seconds=10, plot_start_sec=0)

**Notes on using this with your own / new data (not just MIT-BIH):**
- Your signal must be a single-lead ECG sampled at **360 Hz** (or resample to 360 Hz first) — the beat window sizes and RR features are tuned to this.
- Pass a raw 1D numpy array through `bandpass_filter()` + `normalize()` first, then reuse the beat-detection + prediction logic inside `predict_record` (skip the `wfdb` loading part, which is MIT-BIH-specific).

## 12. Edge Deployment — Export the Two-Input Model to TFLite
Same as before, but the model now takes two inputs (waveform + RR features), which TFLite handles natively.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('ecg_model_v2.tflite', 'wb') as f:
    f.write(tflite_model)

size_kb = os.path.getsize('ecg_model_v2.tflite') / 1024
print(f"TFLite model size: {size_kb:.1f} KB")

# Verify predictions match
interpreter = tf.lite.Interpreter(model_path='ecg_model_v2.tflite')
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# input_details order may vary; match by name
for detail in input_details:
    if 'waveform' in detail['name']:
        wave_idx = detail['index']
    else:
        rr_idx = detail['index']

interpreter.set_tensor(wave_idx, X_test[0:1].astype(np.float32))
interpreter.set_tensor(rr_idx, RR_test[0:1].astype(np.float32))
interpreter.invoke()
tflite_pred = interpreter.get_tensor(output_details[0]['index'])

keras_pred = model.predict({'waveform': X_test[0:1], 'rr_features': RR_test[0:1]}, verbose=0)
print("Keras prediction:", np.argmax(keras_pred), "| TFLite prediction:", np.argmax(tflite_pred))

## 13. Summary
- **Practical output**: `predict_record(record_name)` — the ready-to-use function for "give it any record, tell me the heartbeat types."
- **Fixes applied vs. v1**: RR-interval timing features, oversampling, focal loss — all specifically targeting the S/F recall problem.
- **Two views available**: full 5-class (N/S/V/F/Q) and simplified binary (Normal/Abnormal) — use whichever matches how you want to present results.
- **Deployment-ready**: `ecg_model_v2.tflite` is a quantized, two-input model exportable to a mobile/edge app.

**Compare your new classification report to the v1 one you already have** — S and F recall should be meaningfully higher now. If they're still weak, the next lever to pull is increasing `target_ratio` in the oversampling step (e.g., 0.5 → 0.8) or `gamma` in the focal loss (2.0 → 3.0) to push even harder on the rare classes.